# 03. Guardrails and Human in the Loop

This notebook is a focused guardrails lesson. It shows how a guardrail checks whether the input is safe to process, and how the workflow stops or routes to a person when the text is uncertain.

## Why this matters in DH
In digital humanities, uncertainty is not always a failure. It can be part of the evidence. A blurred place name, a conflicting date, or a doubtful transcription often needs scholarly judgment.

The goal is not to eliminate ambiguity. The goal is to make ambiguity visible and actionable.

## Concepts
- Guardrails
- Human review
- Pass/fail checkpoints
- Validation before action
- Evidence-based decision making

Relevant docs: [Guardrails](https://openai.github.io/openai-agents-python/guardrails/) and [Human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/).

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
The notebook uses a small set of ambiguous archival-style records stored in `data/` and loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Human in the loop: https://openai.github.io/openai-agents-python/human_in_the_loop/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [1]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

In [2]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

records = [{'id': item['id'], 'text': item['text']} for item in load_documents()]
records

[{'id': 1,
  'text': 'Madrid, 4 April 1871\n\nDear brother,\n\nI have finally found a calm hour to write after the commotion of the exhibition. The galleries were crowded, and several visitors asked about the catalog, which made the day long but worthwhile.\n\nIn 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel. She also said she hoped the spring weather would make the journey easier when she next visited.\n\nWith affection,\nMaria'},
 {'id': 2,
  'text': 'Seville, 18 June 1894\n\nEsteemed colleague,\n\nThe printer in Seville announced a new edition of poems by Antonio Ruiz and asked readers to send subscriptions. The notice was meant to reach both bookshops and local reading circles, since the editor believed the edition might sell quickly.\n\nPlease advise whether the notice should also be sent to the archive copy desk. I have also enclosed a short list of the proofs that still require attention before the issue goes out.\n\n

In [3]:
@dataclass
class ReviewDecision:
    record_id: int
    uncertain: bool
    confidence: float
    notes: str


## Step 1: A tiny review heuristic

We treat low confidence or uncertain OCR as a signal for human review. This helper is the policy: it decides whether a record should be reviewed.

In [4]:
def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

clean_decision = ReviewDecision(record_id=1, uncertain=False, confidence=0.95, notes='clear transcription')
uncertain_decision = ReviewDecision(record_id=3, uncertain=True, confidence=0.52, notes='OCR ambiguity around place name')

print('clean text ->', should_review(clean_decision))
print('uncertain text ->', should_review(uncertain_decision))

clean text -> False
uncertain text -> True


## Step 2: Build a guardrail

A guardrail can reject inputs that clearly need human review before the agent continues. Here, `should_review()` holds the policy, and a small text-to-decision helper applies that policy through the SDK.

You can think of them as automated quality checks that sit in front of or behind the agent.

In [5]:
def review_decision_from_text(input_text: str) -> ReviewDecision:
    if not isinstance(input_text, str):
        return ReviewDecision(record_id=0, uncertain=True, confidence=0.0, notes='input is not text')

    text = input_text.lower()

    uncertainty_terms = ['may be', 'unclear', 'uncertain', 'blurred', 'partially legible'] #Add more terms as needed
    uncertainty_hits = sum(term in text for term in uncertainty_terms)
    suspicious_chars = '^~`_|/[]{}<>'
    suspicious_char_hits = sum(ch in suspicious_chars for ch in input_text)

    digit_letter_switch_hits = sum(
        (input_text[i].isalpha() and input_text[i + 1].isdigit())
        or (input_text[i].isdigit() and input_text[i + 1].isalpha())
        for i in range(len(input_text) - 1)
    )
    replacement_char_hits = input_text.count('\ufffd')

    ocr_hits = 0
    if suspicious_char_hits >= 2:
        ocr_hits += 1
    if digit_letter_switch_hits >= 2:
        ocr_hits += 1
    if replacement_char_hits >= 1:
        ocr_hits += 1

    penalty = 0.2 * uncertainty_hits + 0.1 * ocr_hits  #Adjust weights as needed
    confidence = max(0.0, min(1.0, 1.0 - penalty))
    uncertain = uncertainty_hits > 0 or ocr_hits >= 2

    notes = []
    if uncertainty_hits:
        notes.append('explicit uncertainty language detected')
    if ocr_hits:
        notes.append('generic ocr-noise signals detected')
    if not notes:
        notes.append('clear enough to process')

    return ReviewDecision(
        record_id=0,
        uncertain=uncertain,
        confidence=round(confidence, 2),
        notes='; '.join(notes),
    )

def uncertainty_guardrail(ctx, agent, input_text):
    decision = review_decision_from_text(input_text)
    if should_review(decision):
        return GuardrailFunctionOutput(output_info=decision.notes, tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
guardrail

InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x7e98730c6980>, name='uncertainty_guardrail', run_in_parallel=True)

## Step 3: Agent with guardrail

If the text looks uncertain, the agent should stop and route to human review rather than
proceeding with a guess. The guardrail turns the review policy into a runtime stop signal —
it is the mechanism that makes the policy enforceable.

For archival workflows, that protects against fabricated precision and makes the pipeline
more trustworthy: uncertain inputs are never silently processed.

In [6]:
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
    model=DEFAULT_MODEL,
)

review_agent

Agent(name='Review Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Assess whether the text can be safely processed. If it is uncertain, return a short note asking for human review.', prompt=None, handoffs=[], model='gpt-5.4-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[InputGuardrail(guardrail_function=<function uncertainty_guardrail at 0x7e98730c6980>, name='uncertainty_guardrail', run_in_parallel=True)], output_guardrails=[], output_type=None, hooks=None, tool_use_behavio

## Step 4: Checking texts

Run both records through the same helper. If the guardrail passes, the run continues normally. If it triggers, we show an interactive human-review widget.

> **Tip — handling rate limits:** For more resilient execution, replace `await Runner.run(...)` with the `run_with_retry()` helper introduced at the end of notebook 01. It adds automatic exponential backoff for 429 and 5xx errors with no changes to the rest of your code.

In [7]:
import asyncio
import ipywidgets as widgets
from IPython.display import display

In [8]:
HUMAN_REVIEW_ACTIONS: dict[int, str] = {}

def human_review_widget(record: dict, decision: ReviewDecision) -> int:
    """Display an interactive review panel and return record id immediately.

    VS Code notebooks can block on awaited widget callbacks; this pattern stays non-blocking.
    After the user clicks OK, the choice is stored in HUMAN_REVIEW_ACTIONS[record_id].
    """
    record_id = record['id']

    header = widgets.HTML(f"<b>Human Review — Record {record['id']}</b>")
    doc_display = widgets.Textarea(
        value=record['text'],
        disabled=True,
        layout=widgets.Layout(width='100%', height='100px'),
    )
    info = widgets.HTML(
        f"Confidence: <b>{decision.confidence}</b> &nbsp;|&nbsp; Notes: {decision.notes}"
    )
    action_choice = widgets.RadioButtons(
        options=[('Approve', 'approved'), ('Send to archive', 'archived')],
        value='approved',
        description='Action:',
    )
    ok_btn = widgets.Button(description='OK', button_style='primary', icon='check')
    status_label = widgets.Label('Choose an action, then click OK.')
    panel = widgets.VBox([header, info, doc_display, action_choice, ok_btn, status_label])

    def on_ok(_):
        action_choice.disabled = True
        ok_btn.disabled = True
        HUMAN_REVIEW_ACTIONS[record_id] = action_choice.value
        status_label.value = f"Decision recorded: {action_choice.value}"

    ok_btn.on_click(on_ok)
    display(panel)
    return record_id

def get_human_decision(record_id: int) -> str | None:
    return HUMAN_REVIEW_ACTIONS.get(record_id)

In [9]:
from agents import InputGuardrailTripwireTriggered

async def run_case(label: str, record: dict):
    text = record['text']
    decision = review_decision_from_text(text)
    print(f"\n--- {label} (record {record['id']}) ---")
    print('Policy decision:', decision)
    try:
        with trace(f'DH guardrails {label}'):
            result = await Runner.run(review_agent, text)
        print('Pipeline output:', result.final_output)
    except InputGuardrailTripwireTriggered:
        print('Guardrail triggered: InputGuardrailTripwireTriggered')
        record_id = human_review_widget(record, decision)
        print(f"Review widget shown for record {record_id}.")
        print(f"After clicking OK, run: get_human_decision({record_id})")

In [10]:
await run_case('clean case', records[0])


--- clean case (record 1) ---
Policy decision: ReviewDecision(record_id=0, uncertain=False, confidence=1.0, notes='clear enough to process')
Pipeline output: Safe to process.


Now run an ambiguous record through the exact same helper. The only difference should be the guardrail outcome and the handoff to human review.

In [ ]:
await run_case('uncertain case', records[-1])


--- uncertain case (record 6) ---
Policy decision: ReviewDecision(record_id=0, uncertain=True, confidence=0.2, notes='explicit uncertainty language detected; generic ocr-noise signals detected')
Guardrail triggered: InputGuardrailTripwireTriggered


## Human-in-the-loop checkpoint

If the guardrail triggers, the next step is not to force a guess. It is to ask a person to resolve the ambiguity and then resume.

## Exercise

Change the threshold so only records with both OCR ambiguity and low confidence are sent to review.

## Solution

Modify `uncertainty_guardrail` to check for two conditions at once, for example `('unclear' in text and confidence < 0.8)`.

---

## Part 2: Output guardrails

So far the guardrail has been an **input guardrail**: it ran *before* the agent and blocked uncertain documents from entering the pipeline.

An **output guardrail** runs *after* the agent produces its result. It checks the quality of what the agent returned before the result is used downstream.

In digital humanities this is especially useful for catching *unsupported claims*: an extraction that lists people, places, or dates but includes no evidence quotes. That is a red flag — the agent may have hallucinated or guessed rather than extracted from the text.

The pattern is the same as input guardrails:

- Write a function that inspects the output and returns a `GuardrailFunctionOutput`.
- Set `tripwire_triggered=True` to reject the output and raise `OutputGuardrailTripwireTriggered`.
- Attach it to the agent via `output_guardrails=[...]`.

In [ ]:
from dataclasses import dataclass, field
from agents import OutputGuardrail, OutputGuardrailTripwireTriggered

@dataclass
class Extraction:
    people: list[str] = field(default_factory=list)
    places: list[str] = field(default_factory=list)
    dates: list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)

## Step 6: Define the output guardrail

The rule: if the extraction contains any claim (a person, place, or date) but has no evidence quotes, something is wrong. The policy function mirrors `should_review` from the input guardrail — it is plain Python with no model call.

In [ ]:
def has_unsupported_claims(extraction: Extraction) -> bool:
    has_claims = any([extraction.people, extraction.places, extraction.dates])
    return has_claims and not extraction.evidence

def evidence_guardrail_fn(ctx, agent, output: Extraction) -> GuardrailFunctionOutput:
    if has_unsupported_claims(output):
        return GuardrailFunctionOutput(
            output_info="Extraction contains claims with no supporting evidence quotes",
            tripwire_triggered=True,
        )
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)

evidence_guardrail = OutputGuardrail(evidence_guardrail_fn, name="evidence_guardrail")

# Check the policy directly with mock extractions (same pattern as should_review above)
good = Extraction(people=["Maria Gomez"], places=["Madrid"], dates=["1871"], evidence=["'Maria Gomez wrote from Madrid'"])
bad  = Extraction(people=["Maria Gomez"], places=["Madrid"], dates=["1871"], evidence=[])

print("good extraction triggers guardrail:", has_unsupported_claims(good))
print("bad  extraction triggers guardrail:", has_unsupported_claims(bad))

## Step 7: Agent with an output guardrail

Attach the guardrail to an extraction agent. If the agent returns claims without evidence, the guardrail rejects the result before it reaches any downstream consumer.

In [ ]:
extract_agent = Agent(
    name="Archive Extractor",
    instructions=(
        "Extract people, places, and dates from the historical document. "
        "For every claim you make, include a short direct quote from the text as evidence. "
        "Be conservative: only extract what the text explicitly states."
    ),
    output_type=Extraction,
    output_guardrails=[evidence_guardrail],
    model=DEFAULT_MODEL,
)

extract_agent

## Step 8: Run the agent

Run on the clean letter. The agent is instructed to include evidence, so the guardrail should pass. Then observe what happens when we tell the agent to skip evidence — the guardrail trips.

In [ ]:
clean_text = records[0]['text']

print("--- Run 1: normal extraction (evidence expected) ---")
try:
    with trace("output guardrail: clean"):
        result = await Runner.run(extract_agent, clean_text)
    print("Guardrail passed. Extraction:")
    print(result.final_output)
except OutputGuardrailTripwireTriggered as exc:
    print("Guardrail triggered:", exc)

In [ ]:
# An agent that skips evidence — the output guardrail should catch this.
no_evidence_agent = Agent(
    name="Archive Extractor (no evidence)",
    instructions=(
        "Extract people, places, and dates from the historical document. "
        "Do NOT include any evidence quotes — return only the names and dates."
    ),
    output_type=Extraction,
    output_guardrails=[evidence_guardrail],
    model=DEFAULT_MODEL,
)

print("--- Run 2: extraction without evidence (guardrail should trip) ---")
try:
    with trace("output guardrail: no evidence"):
        result = await Runner.run(no_evidence_agent, clean_text)
    print("Guardrail passed. Extraction:")
    print(result.final_output)
except OutputGuardrailTripwireTriggered as exc:
    print("Guardrail triggered — unsupported claims rejected:", exc)

## Summary: input vs. output guardrails

| | Input guardrail | Output guardrail |
|---|---|---|
| **Runs** | Before the agent | After the agent |
| **Inspects** | The incoming text | The agent's result |
| **Catches** | Bad / uncertain input | Bad / unsupported output |
| **Exception** | `InputGuardrailTripwireTriggered` | `OutputGuardrailTripwireTriggered` |
| **DH use case** | Block noisy OCR before extraction | Reject extractions with no evidence |

## Exercise

Extend `evidence_guardrail_fn` so it also trips when any person's name in the extraction is shorter than two characters (a common OCR artifact). Test it with a mock `Extraction` object that has `people=["M"]`.